# MaxTemp Weather Prediction — Exploratory Data Analysis

**Author:** Ritik Pratap Singh Patel  
**Dataset:** Historical weather records (1970–2022)  
**Goal:** Forecast next-day maximum temperature (TMAX) using historical weather data

---

This notebook covers:
1. Dataset Overview
2. Missing Values Analysis
3. Temperature Distribution & Trends
4. Seasonal Patterns
5. Feature Correlations
6. Model Results Summary

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

# Plot styling
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 14

# Load data
df = pd.read_csv('../data/weather.csv', index_col='DATE', parse_dates=True)
print(f'Dataset shape: {df.shape}')
print(f'Date range: {df.index.min()} to {df.index.max()}')

---
## 1. Dataset Overview

In [ ]:
df.head()

In [ ]:
print('Columns:', df.columns.tolist())
print('\nData Types:')
print(df.dtypes)
print(f'\nTotal records: {len(df):,}')
print(f'Total features: {df.shape[1]}')

In [ ]:
df.describe().round(2)

---
## 2. Missing Values Analysis

In [ ]:
null_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
null_pct = null_pct[null_pct > 0]

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#d73027' if v > 5 else '#1a9850' for v in null_pct.values]
ax.barh(null_pct.index, null_pct.values, color=colors)
ax.axvline(x=5, color='black', linestyle='--', linewidth=1, label='5% threshold')
ax.set_xlabel('Missing Values (%)')
ax.set_title('Missing Values by Column (Red = dropped, Green = kept)')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Columns dropped (>5% null): {(null_pct > 5).sum()}')
print(f'Columns retained (<5% null): {(null_pct <= 5).sum() + (df.isnull().sum() == 0).sum()}')

---
## 3. Temperature Distribution & Trends

After cleaning, we work with the retained columns. Let's focus on the key temperature variables.

In [ ]:
# Apply same cleaning as the model
null_pct_all = df.isnull().sum() / len(df)
valid_cols = df.columns[null_pct_all < 0.05]
weather = df[valid_cols].copy()
weather.columns = weather.columns.str.lower()
weather = weather.ffill()

print('Retained columns:', weather.columns.tolist())

In [ ]:
# TMAX distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(weather['tmax'].dropna(), bins=50, color='#e74c3c', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Max Temperature (°C)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Daily Max Temperature (TMAX)')
axes[0].axvline(weather['tmax'].mean(), color='black', linestyle='--', label=f"Mean: {weather['tmax'].mean():.1f}°C")
axes[0].legend()

axes[1].hist(weather['tmin'].dropna(), bins=50, color='#3498db', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('Min Temperature (°C)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Daily Min Temperature (TMIN)')
axes[1].axvline(weather['tmin'].mean(), color='black', linestyle='--', label=f"Mean: {weather['tmin'].mean():.1f}°C")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# TMAX over time
fig, ax = plt.subplots(figsize=(14, 5))
weather['tmax'].plot(ax=ax, color='#e74c3c', alpha=0.4, linewidth=0.5, label='Daily TMAX')
weather['tmax'].rolling(365).mean().plot(ax=ax, color='#c0392b', linewidth=2, label='365-day rolling mean')
ax.set_xlabel('Year')
ax.set_ylabel('Max Temperature (°C)')
ax.set_title('Daily Max Temperature Over Time (1970–2022)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Long-term warming trend (yearly average)
yearly_avg = weather['tmax'].resample('Y').mean()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(yearly_avg.index.year, yearly_avg.values, color='#e74c3c', marker='o', markersize=4, linewidth=1.5)
z = np.polyfit(yearly_avg.index.year, yearly_avg.values, 1)
p = np.poly1d(z)
ax.plot(yearly_avg.index.year, p(yearly_avg.index.year), color='black', linestyle='--', linewidth=1.5, label=f'Trend: {z[0]:.3f}°C/year')
ax.set_xlabel('Year')
ax.set_ylabel('Average Max Temperature (°C)')
ax.set_title('Long-Term Warming Trend in Max Temperature (1970–2022)')
ax.legend()
plt.tight_layout()
plt.show()

---
## 4. Seasonal Patterns

In [ ]:
# Monthly average temperatures
monthly_avg = weather.groupby(weather.index.month)[['tmax', 'tmin']].mean()
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(month_names, monthly_avg['tmax'].values, color='#e74c3c', marker='o', linewidth=2, label='Avg TMAX')
ax.plot(month_names, monthly_avg['tmin'].values, color='#3498db', marker='o', linewidth=2, label='Avg TMIN')
ax.fill_between(month_names, monthly_avg['tmin'].values, monthly_avg['tmax'].values, alpha=0.1, color='gray')
ax.set_xlabel('Month')
ax.set_ylabel('Temperature (°C)')
ax.set_title('Average Monthly Temperature Range (1970–2022)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Season box plots
def get_season(month):
    if month in [12, 1, 2]:  return 'Winter'
    elif month in [3, 4, 5]: return 'Spring'
    elif month in [6, 7, 8]: return 'Summer'
    else:                    return 'Fall'

weather['season'] = weather.index.month.map(get_season)

fig, ax = plt.subplots(figsize=(10, 6))
season_order = ['Winter', 'Spring', 'Summer', 'Fall']
season_colors = ['#3498db', '#2ecc71', '#e74c3c', '#e67e22']
weather.boxplot(column='tmax', by='season', ax=ax,
                positions=[season_order.index(s) for s in season_order],
                patch_artist=True,
                boxprops=dict(facecolor='lightblue'),
                medianprops=dict(color='red', linewidth=2))
ax.set_xticklabels(season_order)
ax.set_xlabel('Season')
ax.set_ylabel('Max Temperature (°C)')
ax.set_title('Max Temperature Distribution by Season')
plt.suptitle('')
plt.tight_layout()
plt.show()

In [ ]:
# Snow depth over time
fig, ax = plt.subplots(figsize=(14, 4))
weather['snwd'].plot(ax=ax, color='#3498db', alpha=0.6, linewidth=0.5)
ax.set_xlabel('Year')
ax.set_ylabel('Snow Depth')
ax.set_title('Snow Depth Over Time (1970–2022)')
plt.tight_layout()
plt.show()

---
## 5. Feature Correlations

In [ ]:
# Correlation heatmap of raw features
numeric_cols = weather.select_dtypes(include=np.number).drop(columns=['season'], errors='ignore')
corr = numeric_cols.corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, linewidths=0.5, ax=ax,
            annot_kws={'size': 10})
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation with TMAX specifically
tmax_corr = corr['tmax'].drop('tmax').sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#d73027' if v < 0 else '#1a9850' for v in tmax_corr.values]
ax.barh(tmax_corr.index, tmax_corr.values, color=colors)
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_xlabel('Correlation with TMAX')
ax.set_title('Feature Correlation with Max Temperature (TMAX)')
plt.tight_layout()
plt.show()

In [ ]:
# TMIN vs TMAX scatter
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(weather['tmin'], weather['tmax'], alpha=0.1, s=1, color='#e74c3c')
ax.set_xlabel('Min Temperature (°C)')
ax.set_ylabel('Max Temperature (°C)')
ax.set_title('TMIN vs TMAX Relationship')
z = np.polyfit(weather['tmin'].dropna(), weather['tmax'].dropna(), 1)
p = np.poly1d(z)
x_line = np.linspace(weather['tmin'].min(), weather['tmin'].max(), 100)
ax.plot(x_line, p(x_line), color='black', linewidth=2, label=f'Trend line')
ax.legend()
plt.tight_layout()
plt.show()

---
## 6. Model Results Summary

The following results are from the trained model in `src/maxtempweatherpredict.py`.

In [ ]:
# Model comparison table
results = {
    'Model':    ['Ridge', 'Random Forest', 'XGBoost', 'LightGBM'],
    'MAE (°C)': [4.77,    5.02,           4.84,      6.89],
    'MSE (°C²)':[37.26,   41.44,          38.50,     71.07],
    'RMSE (°C)':[6.10,    6.44,           6.21,      8.43],
    'R²':       [0.8759,  0.8620,         0.8717,    0.7632],
    'MAPE (%)': [8.88,    9.32,           9.03,      13.18]
}
results_df = pd.DataFrame(results).set_index('Model')
results_df.style.highlight_min(subset=['MAE (°C)', 'MSE (°C²)', 'RMSE (°C)', 'MAPE (%)'], color='lightgreen') \
               .highlight_max(subset=['R²'], color='lightgreen')

In [ ]:
# Model comparison bar chart
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = ['#2ecc71', '#e74c3c', '#e74c3c', '#e74c3c']

for ax, metric in zip(axes, ['MAE (°C)', 'RMSE (°C)', 'R²']):
    bars = ax.bar(results_df.index, results_df[metric],
                  color=['#2ecc71' if i == 0 else '#e74c3c' for i in range(len(results_df))],
                  edgecolor='white')
    ax.set_title(f'Model Comparison — {metric}')
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=15)
    for bar, val in zip(bars, results_df[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.2f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Best Model: Ridge Regression', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Seasonal performance breakdown
seasonal = {
    'Season':   ['Winter', 'Spring', 'Summer', 'Fall'],
    'MAE (°C)': [5.44,     5.46,     3.82,     4.36],
    'RMSE (°C)':[6.78,     6.95,     4.99,     5.47],
    'R²':       [0.4846,   0.6535,   0.4301,   0.7698]
}
seasonal_df = pd.DataFrame(seasonal).set_index('Season')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
season_colors = ['#3498db', '#2ecc71', '#e74c3c', '#e67e22']

axes[0].bar(seasonal_df.index, seasonal_df['MAE (°C)'], color=season_colors, edgecolor='white')
axes[0].set_title('MAE by Season (Best Model: Ridge)')
axes[0].set_ylabel('MAE (°C)')
for i, (idx, val) in enumerate(seasonal_df['MAE (°C)'].items()):
    axes[0].text(i, val + 0.05, f'{val:.2f}', ha='center', fontsize=10)

axes[1].bar(seasonal_df.index, seasonal_df['R²'], color=season_colors, edgecolor='white')
axes[1].set_title('R² by Season (Best Model: Ridge)')
axes[1].set_ylabel('R²')
for i, (idx, val) in enumerate(seasonal_df['R²'].items()):
    axes[1].text(i, val + 0.01, f'{val:.4f}', ha='center', fontsize=10)

plt.suptitle('Winter and Summer show weakest R² — target for v2 improvements', fontsize=11, style='italic')
plt.tight_layout()
plt.show()

---
## Key Findings

1. **Best Model**: Ridge Regression with alpha=100.0 (tuned via GridSearchCV)
   - MAE: 4.77°C | RMSE: 6.10°C | R²: 0.8759 | MAPE: 8.88%

2. **Strongest Features**: `rolling_3_tmax_pct`, `month_avg_tmax`, `tmin` (from feature importance chart)

3. **Seasonal Weakness**: Winter (R²: 0.48) and Summer (R²: 0.43) are hardest to predict
   - Winter: unpredictable cold snaps
   - Summer: heat waves and sudden drops

4. **Long-term Trend**: Slight warming trend visible across 1970–2022
   - `year` feature added to capture this drift

5. **v2 Improvements Planned**:
   - Season-specific features to improve Winter/Summer R²
   - Snow rolling averages for targeted winter improvement
   - Optuna tuning for Random Forest and XGBoost